# CAED Ablation Experiment Results

Visualizes AUC (ROC AUC) scores across three experiment types:
1. **Model comparison** — 4 datasets × 3 models
2. **Bucketing ablation** — high vs. low similarity bucketing per dataset
3. **Context window ablation** — AUC vs. context window size (best model + high bucketing)

Fill in the data input cells below, then run all cells to regenerate charts.
Use `None` for any run that has not been completed yet — bars/points will be silently omitted.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.axis": "y",
})

DATASETS = ["hospital", "beers", "flights", "rayyan"]

os.makedirs("figures", exist_ok=True)

---
## 1. Model Comparison

Enter AUC scores as a nested dict `{model: {dataset: auc_score}}`.

In [ ]:
# ── USER INPUT ───────────────────────────────────────────────────────────────
model_auc = {
    "gpt-5.4":      {"hospital": None, "beers": None, "flights": None, "rayyan": None},
    "gpt-5.4-mini": {"hospital": None, "beers": None, "flights": None, "rayyan": None},
    "gpt-5.4-nano": {"hospital": None, "beers": None, "flights": None, "rayyan": None},
}
# ── END INPUT ─────────────────────────────────────────────────────────────────

In [ ]:
models = list(model_auc.keys())
n_datasets = len(DATASETS)
n_models = len(models)
bar_width = 0.25
x = np.arange(n_datasets)
COLORS = ["#2196F3", "#FF9800", "#4CAF50"]

fig, ax = plt.subplots(figsize=(9, 5))

for i, (model, color) in enumerate(zip(models, COLORS)):
    scores = [model_auc[model].get(d) for d in DATASETS]
    scores_plot = [s if s is not None else float("nan") for s in scores]
    bars = ax.bar(
        x + i * bar_width, scores_plot,
        width=bar_width, label=model, color=color,
        alpha=0.85, edgecolor="white", linewidth=0.6,
    )
    for bar, val in zip(bars, scores_plot):
        if not np.isnan(val):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{val:.3f}",
                ha="center", va="bottom", fontsize=8.5,
            )

ax.set_xticks(x + bar_width * (n_models - 1) / 2)
ax.set_xticklabels([d.capitalize() for d in DATASETS])
ax.set_xlabel("Dataset")
ax.set_ylabel("AUC Score")
ax.set_title("Model Comparison — AUC by Dataset")
ax.set_ylim(0, 1.10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(title="Model", framealpha=0.8)

plt.tight_layout()
plt.savefig("figures/model_comparison.pdf", bbox_inches="tight")
plt.show()

---
## 2. Bucketing Ablation

Enter AUC scores for high vs. low similarity bucketing per dataset.  
Uncomment the `"baseline"` key to add a third bar (bar width adjusts automatically).

In [ ]:
# ── USER INPUT ───────────────────────────────────────────────────────────────
bucketing_auc = {
    "high_similarity": {"hospital": None, "beers": None, "flights": None, "rayyan": None},
    "low_similarity":  {"hospital": None, "beers": None, "flights": None, "rayyan": None},
    # "baseline":      {"hospital": None, "beers": None, "flights": None, "rayyan": None},
}
# ── END INPUT ─────────────────────────────────────────────────────────────────

In [ ]:
configs = list(bucketing_auc.keys())
n_configs = len(configs)
bar_width = 0.35 if n_configs == 2 else 0.25
x = np.arange(n_datasets)
BUCKET_COLORS = ["#5C6BC0", "#EF5350", "#66BB6A"]

fig, ax = plt.subplots(figsize=(9, 5))

for i, (cfg, color) in enumerate(zip(configs, BUCKET_COLORS)):
    scores = [bucketing_auc[cfg].get(d) for d in DATASETS]
    scores_plot = [s if s is not None else float("nan") for s in scores]
    bars = ax.bar(
        x + i * bar_width, scores_plot,
        width=bar_width, label=cfg.replace("_", " ").title(),
        color=color, alpha=0.85, edgecolor="white", linewidth=0.6,
    )
    for bar, val in zip(bars, scores_plot):
        if not np.isnan(val):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{val:.3f}",
                ha="center", va="bottom", fontsize=8.5,
            )

ax.set_xticks(x + bar_width * (n_configs - 1) / 2)
ax.set_xticklabels([d.capitalize() for d in DATASETS])
ax.set_xlabel("Dataset")
ax.set_ylabel("AUC Score")
ax.set_title("Bucketing Ablation — AUC by Dataset")
ax.set_ylim(0, 1.10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(title="Bucketing Config", framealpha=0.8)

plt.tight_layout()
plt.savefig("figures/bucketing_ablation.pdf", bbox_inches="tight")
plt.show()

---
## 3. Context Window Ablation

Best model + high similarity bucketing. One line per dataset, or collapse to a single `"average"` line.  
Keys are context window sizes (integers). Add or remove entries as needed — x-axis adjusts automatically.

In [ ]:
# ── USER INPUT ───────────────────────────────────────────────────────────────
# Style A: one line per dataset
context_window_auc = {
    "hospital": {5: None, 10: None, 20: None, 40: None},
    "beers":    {5: None, 10: None, 20: None, 40: None},
    "flights":  {5: None, 10: None, 20: None, 40: None},
    "rayyan":   {5: None, 10: None, 20: None, 40: None},
}

# Style B: single averaged line (comment out Style A and uncomment below)
# context_window_auc = {
#     "average": {5: None, 10: None, 20: None, 40: None},
# }
# ── END INPUT ─────────────────────────────────────────────────────────────────

In [ ]:
LINE_COLORS = ["#2196F3", "#FF9800", "#4CAF50", "#AB47BC", "#26C6DA"]
MARKERS = ["o", "s", "^", "D", "v"]

fig, ax = plt.subplots(figsize=(8, 5))

all_sizes = sorted({s for d in context_window_auc.values() for s in d})

for (label, size_to_auc), color, marker in zip(
    context_window_auc.items(), LINE_COLORS, MARKERS
):
    sizes = sorted(k for k, v in size_to_auc.items() if v is not None)
    scores = [size_to_auc[s] for s in sizes]

    if not sizes:
        continue

    ax.plot(
        sizes, scores,
        marker=marker, color=color, linewidth=1.8,
        markersize=6, label=label.capitalize(), zorder=3,
    )
    for x_val, y_val in zip(sizes, scores):
        ax.annotate(
            f"{y_val:.3f}",
            (x_val, y_val),
            textcoords="offset points", xytext=(0, 8),
            ha="center", fontsize=8, color=color,
        )

ax.set_xlabel("Context Window Size")
ax.set_ylabel("AUC Score")
ax.set_title("Context Window Ablation — AUC vs. Window Size")
ax.set_ylim(0, 1.10)
ax.set_xticks(all_sizes)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(title="Dataset", framealpha=0.8)

plt.tight_layout()
plt.savefig("figures/context_window_ablation.pdf", bbox_inches="tight")
plt.show()